# Seminar 10

## Knowledge Distillation

In [ ]:
!pip install transformers==4.45.2 sentence-transformers==3.1.1
!pip install evaluate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.3/245.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207

In [ ]:
student_id = "google/bert_uncased_L-2_H-128_A-2"
teacher_id = "textattack/bert-base-uncased-SST-2"

Check that tokenizers are the same

In [ ]:
from transformers import AutoTokenizer

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)

sample = "This is a basic example, with different words to test."

assert teacher_tokenizer(sample) == student_tokenizer(sample)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
sample = "I love pizza"

assert teacher_tokenizer(sample) == student_tokenizer(sample)

data

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

dataset_id="glue"
dataset_config="sst2"

dataset = load_dataset(dataset_id, dataset_config)
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(teacher_id)

def process(examples):
    tokenized_inputs = tokenizer(
        examples["sentence"], truncation=True, max_length=512
    )
    return tokenized_inputs

tokenized_datasets = dataset.map(process, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label","labels")

tokenized_datasets["test"].features

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

{'sentence': Value(dtype='string', id=None),
 'labels': ClassLabel(names=['negative', 'positive'], id=None),
 'idx': Value(dtype='int32', id=None),
 'input_ids': Sequence(feature=Value(dtype='int32', id=None), length=-1, id=None),
 'token_type_ids': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None),
 'attention_mask': Sequence(feature=Value(dtype='int8', id=None), length=-1, id=None)}

In [ ]:
from transformers import TrainingArguments, Trainer
import torch
import torch.nn as nn
import torch.nn.functional as F

class DistillationTrainingArguments(TrainingArguments):
    def __init__(self, *args, alpha=0.5, temperature=2.0, **kwargs):
        super().__init__(*args, **kwargs)

        #instead of 1 - lambda
        self.alpha = alpha
        self.temperature = temperature

class DistillationTrainer(Trainer):
    def __init__(self, *args, teacher_model=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.teacher = teacher_model
        #Teacher and students are on cpu/gpu
        self._move_model_to_device(self.teacher,self.model.device)
        self.teacher.eval()

    def compute_loss(self, student, inputs, return_outputs=False):

        #L_student
        outputs_student = student(**inputs)
        student_loss = outputs_student.loss

        #res of teacher
        with torch.no_grad():
          outputs_teacher = self.teacher(**inputs)


        #softmax with T, L_distillation
        loss_function = nn.KLDivLoss(reduction="batchmean")
        loss_logits = (loss_function(
            F.log_softmax(outputs_student.logits / self.args.temperature, dim=-1),
            F.softmax(outputs_teacher.logits / self.args.temperature, dim=-1)) * (self.args.temperature ** 2))
        #final loss
        loss = self.args.alpha * student_loss + (1. - self.args.alpha) * loss_logits
        return (loss, outputs_student) if return_outputs else loss

In [ ]:
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding


labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label


training_args = DistillationTrainingArguments(
    num_train_epochs=7,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    fp16=True,
    learning_rate=6e-5,
    seed=0,
    output_dir="distillation",
    logging_strategy="epoch", # to get more information to TB
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="tensorboard",
    # distilation parameters
    alpha=0.5,
    temperature=4.0
    )


data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


teacher_model = AutoModelForSequenceClassification.from_pretrained(
    teacher_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)


student_model = AutoModelForSequenceClassification.from_pretrained(
    student_id,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google/bert_uncased_L-2_H-128_A-2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
teacher_model.num_parameters()

109483778

In [ ]:
student_model.num_parameters()

4386178

In [ ]:
teacher_model.num_parameters() / student_model.num_parameters()

24.961088674467838

In [ ]:
!pip install evaluate

In [ ]:
from evaluate import load
import numpy as np

# define metrics and metrics function
accuracy_metric = load( "accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    return {
        "accuracy": acc["accuracy"],
    }

In [ ]:
trainer = DistillationTrainer(
    student_model,
    training_args,
    teacher_model=teacher_model,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,1.443100,1.146398,0.792431


KeyboardInterrupt: 

In [ ]:
31 * 60 / 18

103.33333333333333

## Quantization

In [ ]:
import torch

class M(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = torch.nn.Linear(4, 4)

    def forward(self, x):
        x = self.fc(x)
        return x


model_fp32 = M()

model_int8 = torch.ao.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},  # a set of layers to dynamically quantize
    dtype=torch.qint8)  # the target dtype for quantized weights


input_fp32 = torch.randn(4, 4, 4, 4)
res = model_int8(input_fp32)

In [ ]:
model_fp32.fc.weight.dtype

torch.float32

In [ ]:
model_int8.fc.weight().dtype

torch.qint8

In [ ]:
model_int8(input_fp32)[0][0]

tensor([[-0.5778, -0.8731,  1.2256,  0.3517],
        [-0.5516, -0.1853,  0.6113, -0.7420],
        [ 1.5896, -0.3849, -0.2877,  0.5785],
        [ 1.5851, -1.6432,  0.1357,  0.4307]])

In [ ]:
model_fp32(input_fp32)[0][0]

tensor([[-0.5735, -0.8831,  1.2303,  0.3502],
        [-0.5462, -0.1728,  0.6044, -0.7509],
        [ 1.5861, -0.4018, -0.2802,  0.5938],
        [ 1.5781, -1.6371,  0.1355,  0.4281]], grad_fn=<SelectBackward0>)

## LoRA

Sample eample of LoRA from [tutorial](https://pytorch.org/torchtune/0.4/tutorials/lora_finetune.html)

In [ ]:
import torch
from torch import nn

class LoRALinear(nn.Module):
  def __init__(
    self,
    in_dim: int,
    out_dim: int,
    rank: int,
    alpha: float
  ):
    super().__init__()
    # These are the weights from the original pretrained model
    self.linear = nn.Linear(in_dim, out_dim, bias=False)

    # These are the new LoRA params. In general rank << in_dim, out_dim
    self.lora_a = nn.Linear(in_dim, rank, bias=False)
    self.lora_b = nn.Linear(rank, out_dim, bias=False)

    # Rank and alpha are commonly-tuned hyperparameters
    self.rank = rank
    self.alpha = alpha


    # The original params are frozen, and only LoRA params are trainable.
    self.linear.weight.requires_grad = False
    self.lora_a.weight.requires_grad = True
    self.lora_b.weight.requires_grad = True

  def forward(self, x: torch.Tensor) -> torch.Tensor:
    # This would be the output of the original model
    frozen_out = self.linear(x)

    # lora_a projects inputs down to the much smaller self.rank,
    # then lora_b projects back up to the output dimension
    lora_out = self.lora_b(self.lora_a(x))

    # Finally, scale by the alpha parameter (normalized by rank)
    # and add to the original model's outputs
    return frozen_out + (self.alpha / self.rank) * lora_out

In [ ]:
l = LoRALinear(in_dim = 100,
    out_dim = 200,
    rank = 5,
    alpha = 1)

In [ ]:
#no grad
l.lora_a.weight.grad, l.linear.weight.grad

(None, None)

In [ ]:
input = torch.randn(8, 100)
l(input).shape

torch.Size([8, 200])

In [ ]:
l.linear.weight

Parameter containing:
tensor([[-0.0461,  0.0366, -0.0911,  ..., -0.0075, -0.0596,  0.0659],
        [ 0.0716,  0.0439, -0.0191,  ..., -0.0108,  0.0698,  0.0196],
        [-0.0434, -0.0212, -0.0336,  ...,  0.0518, -0.0574,  0.0684],
        ...,
        [ 0.0734, -0.0984, -0.0614,  ..., -0.0241, -0.0586,  0.0347],
        [-0.0756,  0.0997,  0.0374,  ...,  0.0063,  0.0182,  0.0200],
        [ 0.0520,  0.0948, -0.0148,  ..., -0.0125,  0.0396,  0.0397]])

In [ ]:
l.lora_a.weight

Parameter containing:
tensor([[-9.6061e-02,  2.1875e-02, -5.7947e-02, -7.1107e-02,  9.2177e-02,
          3.1203e-02, -2.6312e-03, -6.3715e-03, -8.0794e-02,  4.1306e-02,
          4.8830e-02,  2.5378e-02,  6.7480e-02, -9.0064e-02, -2.4844e-03,
         -5.2810e-02, -5.1469e-02,  2.9085e-02, -1.2664e-02,  4.7804e-02,
         -5.5797e-02,  7.1353e-02,  9.4586e-02, -8.6629e-02,  6.3589e-02,
         -1.0987e-02, -1.5587e-03,  4.1777e-02,  8.6642e-03, -3.2101e-02,
          5.7200e-02,  2.6291e-02,  5.7288e-02, -1.6251e-02,  4.7508e-02,
          1.5544e-02, -9.0908e-02, -7.2360e-03,  7.1015e-02, -9.6398e-02,
         -9.0636e-02, -9.5219e-02,  9.5801e-02, -4.3064e-02, -4.9587e-02,
         -3.6388e-02,  7.6211e-02, -3.2581e-02, -8.9136e-02,  3.7001e-02,
          3.9806e-02, -2.8935e-02,  4.7723e-02, -3.7376e-02,  1.8203e-02,
          3.9763e-02, -7.3347e-02, -8.3479e-02, -6.2022e-02, -2.4372e-02,
          3.0558e-02, -4.3462e-02,  4.3430e-02, -2.4443e-02, -4.3101e-02,
          9.2503

In [ ]:
loss = l(input).sum()
loss.backward()

In [ ]:
l.linear.weight.grad

In [ ]:
l.lora_a.weight.grad

tensor([[-8.1828e-03,  4.9226e-01,  4.5623e-01,  1.9899e+00,  1.6879e+00,
         -1.9900e+00, -3.2140e-01, -1.4786e+00,  3.0771e-01, -1.3105e+00,
          2.0064e-01, -1.3550e+00, -9.1581e-01, -8.8062e-01,  5.4165e-01,
         -4.0654e-02, -1.3720e-01, -1.5575e+00,  3.1574e-01,  7.2257e-01,
          5.0164e-01, -1.3442e+00, -2.3309e+00, -1.1975e+00,  1.4308e+00,
          1.3442e+00, -8.7149e-01,  7.7565e-01,  4.5282e-01, -3.5017e-02,
         -4.0866e-02, -1.3587e-03,  6.7595e-01, -4.5002e-01, -6.1100e-01,
          4.9539e-01, -5.5540e-01,  4.2137e-01,  1.7584e+00, -4.9727e-01,
         -2.0213e-01, -3.3578e-02, -2.2678e+00,  2.6418e+00,  2.1101e+00,
          1.3789e+00,  1.5920e+00,  7.1215e-01,  3.0881e-01,  4.2541e-01,
          6.8200e-01,  6.3039e-01, -1.6429e-01,  2.1614e+00, -1.8808e+00,
         -7.9946e-01,  2.2911e-01, -9.1722e-01, -9.2038e-01, -1.0207e+00,
          1.2892e-01,  4.8746e-01,  2.1995e-01, -2.1877e+00,  9.7951e-01,
         -1.1843e+00,  2.3111e-01, -3.

Fine-tune BERT model

In [ ]:
!pip install loralib
!pip install datasets
!pip install transformers[sentencepiece]
!pip install evaluate

In [ ]:
import numpy as np
import loralib as lora
import torchvision.models as models
import torch.nn.functional as F
import torch
from torch import nn
from matplotlib import pyplot as plt
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset

In [ ]:
def linear_to_lora_linear(linear, r=6):
    bias = linear.bias is not None
    lora_layer = lora.Linear(
        in_features=linear.in_features,
        out_features=linear.out_features,
        bias=bias,
        #rank
        r=r,
    )

    lora_layer.weight = nn.Parameter(linear.weight.detach())

    if bias:
        lora_layer.bias = nn.Parameter(linear.bias.detach())

    return lora_layer


def to_lora(model, r=6):
    if isinstance(model, nn.ModuleList) or isinstance(model, nn.Sequential):
        for i, elem in enumerate(model):
            if isinstance(model[i], nn.Linear):
                model[i] = linear_to_lora_linear(model[i], r)
            else:
                if isinstance(model, nn.Module):
                    to_lora(model[i])

    else:
        for key in dir(model):
            attr = getattr(model, key)
            if attr is model:
              continue
            if isinstance(attr, nn.Linear):
                setattr(model, key, linear_to_lora_linear(attr, r))
                continue
            if not isinstance(attr, nn.Module):
                continue

            to_lora(attr)

In [ ]:
from datasets import DatasetDict
dataset = load_dataset('tweet_eval', 'irony')

model_name = 'bert-base-cased'
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model = model.cuda()
to_lora(model)
tokenizer = AutoTokenizer.from_pretrained(model_name)
datasets = dataset.map(lambda d: tokenizer(d['text'], truncation=True, padding=True), batched=True)

train-00000-of-00001.parquet:   0%|          | 0.00/183k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/54.0k [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/61.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2862 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/784 [00:00<?, ? examples/s]

...

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/2862 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Map:   0%|          | 0/955 [00:00<?, ? examples/s]

In [ ]:
args = TrainingArguments(
    'lora_bert',
    evaluation_strategy = 'epoch',
    save_strategy = 'no',
    load_best_model_at_end=False,
    weight_decay=0,
    report_to='none',
    learning_rate=5e-3,
    metric_for_best_model='f1',
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
)

import evaluate
metric = evaluate.load('f1')

def compute_metrics(pred):
    logits, labels = pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average='micro')

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1545: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=datasets['train'],
    eval_dataset=datasets['validation'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,No log,0.703564,0.522513
2,No log,0.713665,0.522513
3,0.904600,0.697441,0.477487


TrainOutput(global_step=537, training_loss=0.8912325740082526, metrics={'train_runtime': 181.4264, 'train_samples_per_second': 47.325, 'train_steps_per_second': 2.96, 'total_flos': 1388377756045944.0, 'train_loss': 0.8912325740082526, 'epoch': 3.0})

Llama + LoRA is implemented in library. It is possible to import it

In [ ]:
!pip install torchtune
!pip install torchao

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 7.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 810.3/810.3 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 74.2 MB/s eta 0:00:00
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144555 sha256=ec3cfa4d5af5d2a5aebcbcb0cbbf2f3663c0420922c9d88ee5a145e3090b6470
  Stored in directory: /root/.cache/pip/wheels/1a/97/32/461f837398029ad76911109f07047fde1d7b661a147c7c56d1
Successfully built antlr4-python3-runtime


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 63.0 MB/s eta 0:00:00


In [ ]:
from torchtune.models.llama2 import llama2_7b, lora_llama2_7b

lora_model = lora_llama2_7b(lora_attn_modules=["q_proj", "k_proj", "v_proj", "output_proj"])

In [ ]:
from torchtune.modules.peft.peft_utils import get_adapter_params, set_trainable_params


lora_params = get_adapter_params(lora_model)

set_trainable_params(lora_model, lora_params)

total_params = sum([p.numel() for p in lora_model.parameters()])
trainable_params = sum([p.numel() for p in lora_model.parameters() if p.requires_grad])
print(
  f"""
  {total_params} total params,
  {trainable_params}" trainable params,
  {(100.0 * trainable_params / total_params):.2f}% of all params are trainable.
  """
)

Train using tune:

In [ ]:
!tune run --nnodes 1 --nproc_per_node 2 lora_finetune_distributed --config llama2/7B_lora

## Sources

https://github.com/philschmid/knowledge-distillation-transformers-pytorch-sagemaker/blob/master/knowledge-distillation.ipynb

https://github.com/bayesgroup/HSE_ML_research_seminar/blob/master/2021-2022/182/11%20-%20LoRA/lora.ipynb

https://pytorch.org/torchtune/0.4/tutorials/lora_finetune.html